%md
# Ingerir Dados — Orquestrador de Ingestão (Landing → Bronze)

Ponto de entrada único que ingere as 11 tabelas de evento diário dos 4 sistemas, via `IngestorAutoloader`, da Landing Zone para a Bronze.

Diferente do orquestrador de geração (`gerar_dados.py`), não depende de `data_referencia` — o Autoloader varre toda a Landing Zone do sistema e usa o checkpoint para saber o que já foi processado, independente da data.

Referências: ADR-001 (Landing Zone), ADR-002 (streaming só aqui), ADR-012 (Autoloader).

In [0]:
TABELAS_POR_SISTEMA = {
    "erp": ["erp_lotes_producao", "erp_posicoes_estoque", "erp_notas_expedicao"],
    "crm": ["crm_pedidos", "crm_itens_pedido", "crm_atendimento"],
    "tms": ["tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega"],
    "financeiro": ["financeiro_faturas", "financeiro_contas_receber"],
}

In [0]:
dbutils.widgets.dropdown("sistema", "todos", ["todos", "crm", "erp", "tms", "financeiro"], "Sistema")

In [0]:
sistema_selecionado = dbutils.widgets.get("sistema")
sistemas_a_processar = list(TABELAS_POR_SISTEMA.keys()) if sistema_selecionado == "todos" else [sistema_selecionado]

print(f"Sistemas a processar: {sistemas_a_processar}")

## Execução

Para cada sistema selecionado, ingere todas as suas tabelas de evento diário via `IngestorAutoloader`. A ordem entre sistemas não importa aqui — diferente da geração (ADR-011), ingestão não tem dependência cruzada entre sistemas, só entre Landing e Bronze do mesmo sistema.

In [0]:
from src.ingestao.ingestor_autoloader import IngestorAutoloader

resultados = []
for sistema in sistemas_a_processar:
    for tabela in TABELAS_POR_SISTEMA[sistema]:
        ingestor = IngestorAutoloader(spark=spark, sistema=sistema, tabela=tabela)
        resultado = ingestor.executar()
        resultados.append(resultado)
        print(resultado)

print("\nResumo da ingestão:")
for r in resultados:
    print(f"  {r['tabela']}: {r['status']} — {r['arquivos_processados']} arquivo(s), {r['linhas_processadas']} linha(s)")

In [0]:
resultados = []
for sistema in sistemas_a_processar:
    for tabela in TABELAS_POR_SISTEMA[sistema]:
        ingestor = IngestorAutoloader(spark=spark, sistema=sistema, tabela=tabela)
        resultado = ingestor.executar()
        resultados.append(resultado)
        print(resultado)